<a href="https://colab.research.google.com/github/rayvankaazrifany/skripsi-absa/blob/main/revisi_streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Cell 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# Cell 2: Copy model dari Drive ke Colab
import shutil
shutil.copy('/content/drive/MyDrive/11_aspect_best_multitask_model_2.pt', '/content/11_aspect_best_multitask_model_2.pt')
print("✅ Model siap!")

✅ Model siap!


In [9]:
# Cell 3: Install dependencies
!pip install streamlit pyngrok plotly transformers torch -q
print("✅ Install selesai!")

✅ Install selesai!


In [10]:
# Cell 4: Set ngrok token
!ngrok authtoken 3Dq6zDmh56OGeTVeaEXqPqYruBG_85V1BNHsi2R3BM8LDapoG

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [21]:
# Cell 5: Tulis dashboard.py
dashboard_code = """
import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ast
import torch
import torch.nn as nn
from transformers import BertTokenizer, BertModel

# ================================
# CONFIG
# ================================
st.set_page_config(
    page_title="Dashboard ABSA Layanan Kesehatan",
    page_icon="🏥",
    layout="wide"
)

st.markdown('''
<style>
    @import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;600;700;800&family=Space+Mono&display=swap');
    .stApp { background-color: #f0f9f9; font-family: Plus Jakarta Sans, sans-serif; }
    [data-testid="stSidebar"] { background-color: #e6f5f5 !important; border-right: 1px solid #b2dede; }
    #MainMenu {visibility: hidden;} footer {visibility: hidden;}
    .metric-card { background: #ffffff; border: 1px solid #b2dede; border-radius: 14px; padding: 24px; position: relative; overflow: hidden; margin-bottom: 8px; box-shadow: 0 2px 8px rgba(0,174,172,0.08); }
    .metric-card::after { content: ""; position: absolute; top: 0; left: 0; right: 0; height: 3px; border-radius: 14px 14px 0 0; }
    .metric-card.total::after { background: linear-gradient(90deg, #00AEAC, #1B3A6B); }
    .metric-card.positive::after { background: linear-gradient(90deg, #00AEAC, #00d4d1); }
    .metric-card.negative::after { background: linear-gradient(90deg, #e05a2b, #f0845a); }
    .metric-card.neutral::after { background: linear-gradient(90deg, #C8D400, #e0eb4a); }
    .metric-label { font-size: 0.72rem; font-weight: 700; color: #1B3A6B; text-transform: uppercase; letter-spacing: 1.2px; margin-bottom: 10px; font-family: Space Mono, monospace; }
    .metric-value { font-size: 2.4rem; font-weight: 800; color: #1B3A6B; line-height: 1; margin-bottom: 4px; }
    .metric-pct { font-size: 0.85rem; color: #4a7a8a; }
    .metric-icon { position: absolute; top: 20px; right: 20px; font-size: 1.8rem; opacity: 0.15; }
    .section-title { font-size: 1.05rem; font-weight: 700; color: #1B3A6B; margin: 0 0 4px 0; }
    .section-subtitle { font-size: 0.8rem; color: #4a7a8a; margin: 0 0 16px 0; }
    .insight-card { border-radius: 12px; padding: 20px; margin-bottom: 12px; }
    .insight-best { background: rgba(0,174,172,0.08); border: 1px solid rgba(0,174,172,0.35); }
    .insight-warn { background: rgba(224,90,43,0.08); border: 1px solid rgba(224,90,43,0.35); }
    .insight-title { font-size: 0.7rem; font-weight: 700; text-transform: uppercase; letter-spacing: 1.2px; margin-bottom: 8px; font-family: Space Mono, monospace; }
    .insight-best .insight-title { color: #00AEAC; }
    .insight-warn .insight-title { color: #e05a2b; }
    .insight-aspect { font-size: 1.1rem; font-weight: 700; color: #1B3A6B; margin-bottom: 4px; }
    .insight-desc { font-size: 0.82rem; color: #4a7a8a; line-height: 1.5; }
    .rank-row { display: flex; align-items: center; padding: 12px 16px; border-radius: 10px; margin-bottom: 8px; background: #ffffff; border: 1px solid #b2dede; gap: 14px; box-shadow: 0 1px 4px rgba(0,174,172,0.06); }
    .rank-num { font-family: Space Mono, monospace; font-size: 0.85rem; font-weight: 700; color: #4a7a8a; width: 24px; flex-shrink: 0; }
    .rank-name { font-size: 0.88rem; font-weight: 600; color: #1B3A6B; flex: 1; }
    .rank-score { font-family: Space Mono, monospace; font-size: 0.82rem; font-weight: 700; padding: 3px 10px; border-radius: 20px; }
    .sim-result-card { border-radius: 12px; padding: 20px; margin-bottom: 10px; background: #ffffff; border: 1px solid #b2dede; }
    .sim-sent-positive { border-left: 4px solid #00AEAC; }
    .sim-sent-negative { border-left: 4px solid #e05a2b; }
    .sim-sent-neutral { border-left: 4px solid #C8D400; }
    .badge { display: inline-block; padding: 3px 10px; border-radius: 20px; font-size: 0.75rem; font-weight: 600; margin-right: 6px; margin-top: 6px; }
    .badge-positive { background: rgba(0,174,172,0.12); color: #00AEAC; border: 1px solid rgba(0,174,172,0.3); }
    .badge-negative { background: rgba(224,90,43,0.12); color: #e05a2b; border: 1px solid rgba(224,90,43,0.3); }
    .badge-neutral { background: rgba(200,212,0,0.15); color: #8a9200; border: 1px solid rgba(200,212,0,0.4); }
    .badge-aspect { background: rgba(27,58,107,0.08); color: #1B3A6B; border: 1px solid rgba(27,58,107,0.2); }
</style>
''', unsafe_allow_html=True)

# ================================
# CONSTANTS
# ================================
ASPECT_LABELS = [
    'persyaratan',
    'sistem_mekanisme_prosedur',
    'waktu_penyelesaian',
    'biaya_tarif',
    'produk_spesifikasi_pelayanan',
    'kompetensi_pelaksana',
    'perilaku_pelaksana',
    'penanganan_pengaduan_saran_masukan',
    'sarana_prasarana',
    'empati_dan_komunikasi',
    'keandalan_dan_daya_tangkap',
]
ASPECT_DISPLAY = {
    'persyaratan': 'Persyaratan',
    'sistem_mekanisme_prosedur': 'Sistem & Prosedur',
    'waktu_penyelesaian': 'Waktu Penyelesaian',
    'biaya_tarif': 'Biaya & Tarif',
    'produk_spesifikasi_pelayanan': 'Produk & Spesifikasi Layanan',
    'kompetensi_pelaksana': 'Kompetensi Pelaksana',
    'perilaku_pelaksana': 'Perilaku Pelaksana',
    'penanganan_pengaduan_saran_masukan': 'Penanganan Pengaduan & Saran',
    'sarana_prasarana': 'Sarana & Prasarana',
    'empati_dan_komunikasi': 'Empati & Komunikasi',
    'keandalan_dan_daya_tangkap': 'Keandalan & Daya Tangkap',
}
SENT_EMOJI = {'positive': '😊', 'negative': '😞', 'neutral': '😐'}
SENT_COLOR = {'positive': '#00AEAC', 'negative': '#e05a2b', 'neutral': '#C8D400'}
SENT_LABEL = {'positive': 'Positif', 'negative': 'Negatif', 'neutral': 'Netral'}

# ================================
# MODEL
# ================================
class MultiTaskIndoBERT(nn.Module):
    def __init__(self, num_sentiment=3, num_aspect=12, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained('indobenchmark/indobert-base-p1')
        self.dropout = nn.Dropout(dropout)
        self.sentiment_head = nn.Linear(self.bert.config.hidden_size, num_sentiment)
        self.aspect_head = nn.Linear(self.bert.config.hidden_size, num_aspect)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(out.pooler_output)
        return self.sentiment_head(pooled), self.aspect_head(pooled)

@st.cache_resource
def load_model():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = MultiTaskIndoBERT()
    model.load_state_dict(torch.load('11_aspect_best_multitask_model_2.pt', map_location=device))
    model.eval()
    model.to(device)
    tokenizer = BertTokenizer.from_pretrained('indobenchmark/indobert-base-p1')
    return model, tokenizer, device

def predict(texts, model, tokenizer, device, max_len=128):
    sent_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
    enc = tokenizer(texts, max_length=max_len, padding='max_length',
                    truncation=True, return_tensors='pt')
    with torch.no_grad():
        sl, al = model(enc['input_ids'].to(device), enc['attention_mask'].to(device))
    sents = [sent_map[i] for i in sl.argmax(dim=1).cpu().numpy()]
    aspects = []
    for row in (torch.sigmoid(al) > 0.5).cpu().numpy().astype(int):
        a = [ASPECT_LABELS[i] for i in range(len(ASPECT_LABELS)) if row[i] == 1]
        aspects.append(a if a else ['lainnya'])
    return sents, aspects

# ================================
# METRICS HELPER
# ================================
def compute_metrics(df):
    total = len(df)
    vc = df['sentiment_label'].value_counts()
    pos, neg, neu = vc.get('positive',0), vc.get('negative',0), vc.get('neutral',0)
    asp_sent = {a: {'positive':0,'negative':0,'neutral':0} for a in ASPECT_LABELS}
    for _, row in df.iterrows():
        asps = row['aspect_classification']
        if isinstance(asps, str):
            try: asps = ast.literal_eval(asps)
            except: asps = ['lainnya']
        if not asps: asps = ['lainnya']
        for a in asps:
            if a in asp_sent and row['sentiment_label'] in asp_sent[a]:
                asp_sent[a][row['sentiment_label']] += 1
    scores = {}
    for a, c in asp_sent.items():
        t = sum(c.values())
        scores[a] = (c['positive'] - c['negative']) / t if t > 0 else 0
    return {
        'total': total, 'pos': pos, 'neg': neg, 'neu': neu,
        'pos_pct': pos/total*100, 'neg_pct': neg/total*100, 'neu_pct': neu/total*100,
        'asp_sent': asp_sent, 'scores': scores,
        'best': max(scores, key=scores.get),
        'worst': min(scores, key=scores.get)
    }

def chart_bar(asp_sent):
    labels = [ASPECT_DISPLAY[a] for a in ASPECT_LABELS]
    fig = go.Figure()
    for name, color, key in [
        ('Positif', '#1D9E75', 'positive'),
        ('Netral',  '#B4B2A9', 'neutral'),
        ('Negatif', '#D85A30', 'negative'),
    ]:
        values = [asp_sent[a][key] for a in ASPECT_LABELS]
        fig.add_trace(go.Bar(
            name=name,
            x=labels,
            y=values,
            marker_color=color,
            marker_line_width=0,
        ))
    fig.update_layout(
        barmode='stack',
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(family='Plus Jakarta Sans', color='#4a7a8a', size=10),
        legend=dict(orientation='h', y=1.08, bgcolor='rgba(0,0,0,0)'),
        margin=dict(l=10, r=10, t=40, b=90),
        height=420,
        xaxis=dict(
            gridcolor='rgba(0,0,0,0)',
            tickfont=dict(size=9),
            tickangle=-30,
        ),
        yaxis=dict(gridcolor='rgba(178,222,222,0.4)'),
    )
    return fig

def chart_pie(pos, neg, neu):
    fig = go.Figure(go.Pie(
        labels=['Positif','Negatif','Netral'], values=[pos,neg,neu], hole=0.62,
        marker=dict(colors=['#00AEAC', '#e05a2b', '#C8D400'], line=dict(color='#f0f9f9', width=3)),
        textinfo='percent', hovertemplate='<b>%{label}</b><br>%{value} ulasan<extra></extra>'
    ))
    fig.update_layout(
        paper_bgcolor='rgba(0,0,0,0)', font=dict(family='Plus Jakarta Sans', color='#94a3b8'),
        legend=dict(orientation='v', yanchor='middle', y=0.5, bgcolor='rgba(0,0,0,0)'),
        margin=dict(l=10,r=10,t=10,b=10), height=280,
        annotations=[dict(text=f'<b>{pos+neg+neu}</b><br>Ulasan', x=0.5, y=0.5,
                          font=dict(color='#f1f5f9', size=16, family='Plus Jakarta Sans'), showarrow=False)]
    )
    return fig

# ================================
# SIDEBAR
# ================================
with st.sidebar:
    st.markdown('''
    <div style="padding:16px 0 20px 0;">
        <div style="font-family:Space Mono,monospace;font-size:0.65rem;color:#3b82f6;letter-spacing:2px;text-transform:uppercase;margin-bottom:8px;">ABSA Dashboard</div>
        <div style="font-size:1.2rem;font-weight:800;color:#f1f5f9;">🏥 Layanan Kesehatan</div>
    </div>
    ''', unsafe_allow_html=True)
    st.markdown("---")
    menu = st.radio("", ["📊 Hasil Analisis", "🔍 Simulasi Prediksi"], label_visibility="collapsed")

# ================================
# HEADER
# ================================
st.markdown('''
<div style="background:linear-gradient(135deg,#ffffff,#e6f5f5);border:1px solid #b2dede;border-radius:16px;padding:28px 36px;margin-bottom:24px;box-shadow:0 4px 16px rgba(0,174,172,0.1);">
    <div style="font-family:Space Mono,monospace;font-size:0.65rem;color:#00AEAC;letter-spacing:2px;text-transform:uppercase;margin-bottom:10px;">🏥 Healthcare NLP · IndoBERT Multi-Task · 11 Aspek SKM</div>
    <h1 style="font-size:1.8rem;font-weight:800;color:#1B3A6B;margin:0 0 6px 0;">Dashboard Analisis Sentimen & Aspek</h1>
    <p style="font-size:0.9rem;color:#4a7a8a;margin:0;">RS Mata Cicendo · Kemenkes RI · Aspect-Based Sentiment Analysis</p>
</div>
''', unsafe_allow_html=True)

# ================================
# PAGE: HASIL ANALISIS
# ================================
if menu == "📊 Hasil Analisis":
    uploaded = st.file_uploader("Upload hasil prediksi (CSV/Excel)", type=['csv','xlsx'])

    if uploaded:
        df = pd.read_csv(uploaded) if uploaded.name.endswith('.csv') else pd.read_excel(uploaded)
        def safe_parse(x):
            if isinstance(x, list): return x
            try: return ast.literal_eval(str(x))
            except: return ['lainnya']
        df['aspect_classification'] = df['aspect_classification'].apply(safe_parse)
        st.success(f"✅ {len(df)} data berhasil dimuat!")

        m = compute_metrics(df)

        # Metric Cards
        c1, c2, c3, c4 = st.columns(4)
        CARD_ICON_SVG = {
            'total': '''<svg width="28" height="28" viewBox="0 0 24 24" fill="none" stroke="#1B3A6B" stroke-width="1.6" stroke-linecap="round" stroke-linejoin="round">
                <path d="M21 15a2 2 0 0 1-2 2H7l-4 4V5a2 2 0 0 1 2-2h14a2 2 0 0 1 2 2z"/>
            </svg>''',
            'positive': '''<svg width="28" height="28" viewBox="0 0 24 24" fill="none" stroke="#00AEAC" stroke-width="1.6" stroke-linecap="round" stroke-linejoin="round">
                <path d="M14 9V5a3 3 0 0 0-3-3l-4 9v11h11.28a2 2 0 0 0 2-1.7l1.38-9a2 2 0 0 0-2-2.3H14z"/>
                <path d="M7 22H4a2 2 0 0 1-2-2v-7a2 2 0 0 1 2-2h3"/>
            </svg>''',
            'negative': '''<svg width="28" height="28" viewBox="0 0 24 24" fill="none" stroke="#e05a2b" stroke-width="1.6" stroke-linecap="round" stroke-linejoin="round">
                <path d="M10 15v4a3 3 0 0 0 3 3l4-9V2H5.72a2 2 0 0 0-2 1.7l-1.38 9a2 2 0 0 0 2 2.3H10z"/>
                <path d="M17 2h2.67A2.31 2.31 0 0 1 22 4v7a2.31 2.31 0 0 1-2.33 2H17"/>
            </svg>''',
            'neutral': '''<svg width="28" height="28" viewBox="0 0 24 24" fill="none" stroke="#8a9200" stroke-width="1.6" stroke-linecap="round" stroke-linejoin="round">
                <circle cx="12" cy="12" r="10"/>
                <line x1="8" y1="15" x2="16" y2="15"/>
                <line x1="9" y1="9" x2="9.01" y2="9"/>
                <line x1="15" y1="9" x2="15.01" y2="9"/>
            </svg>''',
        }

        for col, cls, label, val, pct in [
            (c1,'total','Total Ulasan', m['total'], 'Keseluruhan data'),
            (c2,'positive','Sentimen Positif', m['pos'], f"{m['pos_pct']:.1f}% dari total"),
            (c3,'negative','Sentimen Negatif', m['neg'], f"{m['neg_pct']:.1f}% dari total"),
            (c4,'neutral','Sentimen Netral', m['neu'], f"{m['neu_pct']:.1f}% dari total"),
        ]:
            icon_svg = CARD_ICON_SVG[cls]
            with col:
                st.markdown(f'''
                <div class="metric-card {cls}">
                    <div style="display:flex;justify-content:space-between;align-items:flex-start;margin-bottom:12px;">
                        <div class="metric-label">{label}</div>
                        <div style="width:44px;height:44px;border-radius:10px;background:rgba(255,255,255,0.7);display:flex;align-items:center;justify-content:center;flex-shrink:0;">
                            {icon_svg}
                        </div>
                    </div>
                    <div class="metric-value">{val:,}</div>
                    <div class="metric-pct">{pct}</div>
                </div>''', unsafe_allow_html=True)

        st.markdown("<div style='height:20px'></div>", unsafe_allow_html=True)

        # Insight + Pie
        ci, cp = st.columns(2)
        with ci:
            st.markdown('<div class="section-title">💡 Insight Aspek</div><div class="section-subtitle">Aspek terbaik dan yang perlu perhatian</div>', unsafe_allow_html=True)
            for key, cls, emoji, label in [
                (m['best'],'insight-best','⭐','Aspek Terbaik'),
                (m['worst'],'insight-warn','⚠️','Perlu Perhatian')
            ]:
                c = m['asp_sent'][key]
                t = sum(c.values())
                pct_val = (c['positive']/t*100 if t > 0 else 0) if label=='Aspek Terbaik' else (c['negative']/t*100 if t > 0 else 0)
                pct_color = '#34d399' if label=='Aspek Terbaik' else '#f87171'
                pct_label = 'positif' if label=='Aspek Terbaik' else 'negatif'
                st.markdown(f'''
                <div class="insight-card {cls}">
                    <div class="insight-title">{emoji} {label}</div>
                    <div class="insight-aspect">{ASPECT_DISPLAY[key]}</div>
                    <div class="insight-desc">
                        {c['positive']} positif · {c['neutral']} netral · {c['negative']} negatif
                        &nbsp;|&nbsp; <b style="color:{pct_color}">{pct_val:.0f}% {pct_label}</b>
                    </div>
                </div>''', unsafe_allow_html=True)

        with cp:
            st.markdown('<div class="section-title">🥧 Distribusi Sentimen</div><div class="section-subtitle">Keseluruhan ulasan</div>', unsafe_allow_html=True)
            st.plotly_chart(chart_pie(m['pos'],m['neg'],m['neu']), use_container_width=True, config={'displayModeBar':False})

        # Bar Chart
        st.markdown('<div class="section-title">📊 Distribusi Sentimen per Aspek</div><div class="section-subtitle">Positif (hijau) · Netral (abu) · Negatif (merah)</div>', unsafe_allow_html=True)
        st.plotly_chart(chart_bar(m['asp_sent']), use_container_width=True, config={'displayModeBar':False})

        # Peringkat
        st.markdown('<div class="section-title">🏆 Peringkat Aspek</div><div class="section-subtitle">Terpositif ke ternegatif · proporsi dari total keseluruhan ulasan</div>', unsafe_allow_html=True)
        sorted_asp = sorted(m['scores'].items(), key=lambda x: x[1], reverse=True)
        total_all = m['total'] or 1
        html = ''
        for i, (asp, score) in enumerate(sorted_asp):
            c = m['asp_sent'][asp]
            t_asp = sum(c.values()) or 1
            pos_pct = c['positive'] / total_all * 100
            neu_pct = c['neutral']  / total_all * 100
            neg_pct = c['negative'] / total_all * 100
            pos_w   = c['positive'] / total_all * 100
            neu_w   = c['neutral']  / total_all * 100
            neg_w   = c['negative'] / total_all * 100
            total_asp = c['positive'] + c['neutral'] + c['negative']
            html += f'''
            <div style="margin-bottom:18px;">
                <div style="display:flex;justify-content:space-between;align-items:baseline;margin-bottom:4px;">
                    <span style="font-size:0.88rem;font-weight:700;color:#1B3A6B;">{ASPECT_DISPLAY[asp]}</span>
                    <span style="font-size:0.78rem;color:#4a7a8a;font-family:Space Mono,monospace;">{total_asp:,} ulasan</span>
                </div>
                <div style="width:100%;height:10px;border-radius:6px;overflow:hidden;display:flex;background:#e5e7eb;">
                    <div style="width:{pos_w:.2f}%;background:#1D9E75;height:100%;"></div>
                    <div style="width:{neu_w:.2f}%;background:#B4B2A9;height:100%;"></div>
                    <div style="width:{neg_w:.2f}%;background:#D85A30;height:100%;"></div>
                </div>
                <div style="display:flex;gap:20px;margin-top:6px;flex-wrap:wrap;">
                    <span style="font-size:0.75rem;color:#1D9E75;font-family:Space Mono,monospace;">
                        ● Positif &nbsp;<b>{c['positive']:,}</b> &nbsp;<span style="color:#4a7a8a;">({pos_pct:.1f}%)</span>
                    </span>
                    <span style="font-size:0.75rem;color:#888780;font-family:Space Mono,monospace;">
                        ● Netral &nbsp;<b>{c['neutral']:,}</b> &nbsp;<span style="color:#4a7a8a;">({neu_pct:.1f}%)</span>
                    </span>
                    <span style="font-size:0.75rem;color:#D85A30;font-family:Space Mono,monospace;">
                        ● Negatif &nbsp;<b>{c['negative']:,}</b> &nbsp;<span style="color:#4a7a8a;">({neg_pct:.1f}%)</span>
                    </span>
                </div>
            </div>'''
        st.markdown(html, unsafe_allow_html=True)

        with st.expander("📋 Lihat Data Lengkap"):
            fa, fb, fc = st.columns([2, 2, 2])
            with fa:
                sort_by = st.selectbox("Urutkan", ["Skor Tertinggi", "Skor Terendah", "Positif Dulu", "Negatif Dulu"])
            with fb:
                filter_sent = st.multiselect("Filter Sentimen", ['positive', 'negative', 'neutral'],
                                             default=['positive', 'negative', 'neutral'],
                                             format_func=lambda x: SENT_LABEL[x])
            with fc:
                filter_asp = st.multiselect("Filter Aspek", ASPECT_LABELS, default=[],
                                            format_func=lambda x: ASPECT_DISPLAY.get(x, x),
                                            placeholder="Semua aspek")

            disp_df = df.copy()
            if filter_sent:
                disp_df = disp_df[disp_df['sentiment_label'].isin(filter_sent)]
            if filter_asp:
                disp_df = disp_df[disp_df['aspect_classification'].apply(lambda a: any(x in a for x in filter_asp))]

            if sort_by == "Skor Tertinggi":
                disp_df = disp_df.sort_values('sentiment_score', ascending=False)
            elif sort_by == "Skor Terendah":
                disp_df = disp_df.sort_values('sentiment_score', ascending=True)
            elif sort_by == "Positif Dulu":
                order = {'positive': 0, 'neutral': 1, 'negative': 2}
                disp_df = disp_df.assign(_o=disp_df['sentiment_label'].map(order)).sort_values(['_o', 'sentiment_score'], ascending=[True, False]).drop(columns='_o')
            elif sort_by == "Negatif Dulu":
                order = {'negative': 0, 'neutral': 1, 'positive': 2}
                disp_df = disp_df.assign(_o=disp_df['sentiment_label'].map(order)).sort_values(['_o', 'sentiment_score'], ascending=[True, False]).drop(columns='_o')

            tbl = disp_df[['text_translated', 'sentiment_label', 'sentiment_score', 'aspect_classification']].copy()
            tbl.columns = ['Ulasan', 'Sentimen', 'Skor', 'Aspek']
            tbl['Sentimen'] = tbl['Sentimen'].map(lambda x: f"{SENT_EMOJI.get(x,'')} {SENT_LABEL.get(x,x)}")
            tbl['Skor'] = tbl['Skor'].map(lambda x: f"{x:.4f}")
            tbl['Aspek'] = tbl['Aspek'].apply(lambda a: ', '.join([ASPECT_DISPLAY.get(x, x) for x in a]) if isinstance(a, list) else a)
            tbl = tbl.reset_index(drop=True)
            tbl.index += 1

            st.markdown(f"<div style='font-size:0.8rem;color:#64748b;margin-bottom:8px;'>Menampilkan <b style='color:#f1f5f9'>{len(tbl):,}</b> dari {len(df):,} ulasan</div>", unsafe_allow_html=True)
            st.dataframe(tbl, use_container_width=True, height=380)
    else:
        st.markdown('''
        <div style="text-align:center;padding:60px 20px;background:#111827;border:2px dashed #1e3a5f;border-radius:14px;">
            <div style="font-size:3rem;margin-bottom:12px;">📊</div>
            <div style="font-size:1rem;font-weight:600;color:#94a3b8;margin-bottom:8px;">Upload File Hasil Prediksi</div>
            <div style="font-size:0.82rem;color:#475569;">CSV atau Excel dengan kolom: <code style="background:#1e293b;padding:2px 6px;border-radius:4px;color:#7dd3fc;">text_translated, sentiment_label, aspect_classification</code></div>
        </div>''', unsafe_allow_html=True)

# ================================
# PAGE: SIMULASI
# ================================
elif menu == "🔍 Simulasi Prediksi":
    st.markdown('<div class="section-title">🔍 Simulasi Prediksi ABSA</div><div class="section-subtitle">Masukkan ulasan layanan kesehatan untuk diprediksi sentimen dan aspeknya</div>', unsafe_allow_html=True)

    with st.spinner("Memuat model IndoBERT..."):
        try:
            model, tokenizer, device = load_model()
            st.success("✅ Model siap!")
        except Exception as e:
            st.error(f"❌ Gagal memuat model: {e}")
            st.stop()

    input_text = st.text_area(
        "Masukkan ulasan (satu per baris)",
        placeholder="Dokternya sangat ramah dan sabar menjelaskan kondisi saya...\\nAntriannya sangat panjang, saya menunggu 3 jam tanpa kejelasan...\\nFasilitas ruang tunggu cukup bersih dan nyaman.",
        height=160
    )

    col_btn, _ = st.columns([1, 4])
    with col_btn:
        predict_btn = st.button("🔍 Prediksi", type="primary", use_container_width=True)

    if predict_btn and input_text.strip():
        texts = [t.strip() for t in input_text.strip().split('\\n') if t.strip()]
        with st.spinner(f"Memproses {len(texts)} ulasan..."):
            sents, aspects = predict(texts, model, tokenizer, device)

        st.markdown(f"<div style='margin:20px 0 12px 0;font-size:0.85rem;color:#64748b;'>Hasil prediksi untuk <b style='color:#f1f5f9'>{len(texts)} ulasan</b></div>", unsafe_allow_html=True)

        for i, (text, sent, asps) in enumerate(zip(texts, sents, aspects)):
            sent_color = SENT_COLOR[sent]
            sent_bg = f'rgba({("16,185,129" if sent=="positive" else ("239,68,68" if sent=="negative" else "107,114,128"))},0.08)'
            badges_asp = ''.join([f'<span class="badge badge-aspect">{ASPECT_DISPLAY.get(a,a)}</span>' for a in asps])
            st.markdown(f'''
            <div class="sim-result-card sim-sent-{sent}">
                <div style="font-size:0.72rem;color:#64748b;font-family:Space Mono,monospace;margin-bottom:8px;">ULASAN #{i+1}</div>
                <div style="font-size:0.95rem;color:#e2e8f0;margin-bottom:12px;line-height:1.6;">"{text}"</div>
                <div style="display:flex;align-items:center;gap:8px;flex-wrap:wrap;">
                    <span class="badge badge-{sent}" style="font-size:0.8rem;">{SENT_EMOJI[sent]} {sent.capitalize()}</span>
                    {badges_asp}
                </div>
            </div>''', unsafe_allow_html=True)

        # Ringkasan mini
        if len(texts) > 1:
            st.markdown("<div style='height:16px'></div>", unsafe_allow_html=True)
            st.markdown('<div class="section-title">📈 Ringkasan</div>', unsafe_allow_html=True)
            from collections import Counter
            sent_count = Counter(sents)
            asp_count = Counter([a for asp in aspects for a in asp])
            rc1, rc2 = st.columns(2)
            with rc1:
                st.markdown("**Distribusi Sentimen**")
                for s, cnt in sent_count.most_common():
                    color = SENT_COLOR[s]
                    pct = cnt/len(texts)*100
                    st.markdown(f'''
                    <div style="display:flex;align-items:center;gap:10px;margin-bottom:8px;">
                        <span style="width:80px;font-size:0.82rem;color:#94a3b8;">{SENT_EMOJI[s]} {s}</span>
                        <div style="flex:1;height:8px;background:#1e293b;border-radius:4px;">
                            <div style="width:{pct}%;height:100%;background:{color};border-radius:4px;"></div>
                        </div>
                        <span style="font-size:0.82rem;color:#64748b;width:50px;text-align:right;">{cnt} ({pct:.0f}%)</span>
                    </div>''', unsafe_allow_html=True)
            with rc2:
                st.markdown("**Aspek yang Muncul**")
                for asp, cnt in asp_count.most_common():
                    st.markdown(f'<span class="badge badge-aspect">{ASPECT_DISPLAY.get(asp,asp)}</span> <span style="font-size:0.8rem;color:#64748b;">{cnt}x</span><br>', unsafe_allow_html=True)
"""

with open('dashboard.py', 'w') as f:
    f.write(dashboard_code)
print("✅ dashboard.py berhasil ditulis!")

✅ dashboard.py berhasil ditulis!


In [23]:
# Cell 6: Jalankan Streamlit + ngrok
import subprocess, threading, time
from pyngrok import ngrok

# Matikan tunnel lama kalau ada
ngrok.kill()

def run_streamlit():
    subprocess.run([
        'streamlit', 'run', 'dashboard.py',
        '--server.port=8501',
        '--server.headless=true',
        '--server.enableCORS=false'
    ])

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(5)

url = ngrok.connect(8501)
print(f"\n🚀 Dashboard bisa dibuka di:\n{url}")


🚀 Dashboard bisa dibuka di:
NgrokTunnel: "https://dangling-gauze-puppet.ngrok-free.dev" -> "http://localhost:8501"
